In [1]:
import pandas as pd
import numpy as np
from db_queries import get_covariate_estimates, get_location_metadata

In [2]:
data_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_09_23.01/data.parquet"
no_cutoff_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/child_mortality/training_data/2026_09_23.01/data_no_10_yr_cutoff.parquet"

In [3]:
df = pd.read_parquet(data_path)
# df.drop(columns=["index"], inplace=True)
df.head()

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_3_m,age_6_m,age_12_m,age_24_m,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound
0,MWI,132,132,4,8,86,-13.260248,33.648465,92093,218581_132_86_8_2_1990_1_2015_11_310_1_48_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
1,IND,19909,19909,19922,3,27,25.217504,82.598196,72400,467681_19909_27_3_2_2015_9_2020_1_64_1_NA_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
2,GHA,198,198,5,1,28,5.645717,-0.152658,92667,218572_198_28_1_2_2015_6_2017_8_26_1_33_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
3,IDN,75021014,75021014,669,4,14,0.744408,122.637229,71333,20011_75021014_14_4_1_1989_11_2003_3_160_1_36_NA,...,0,0,0,0,0,0,0,age_1_m,0,1
4,ZWE,946,946,17,1,72,-17.877540,31.055500,90799,21163_946_72_1_1_1999_4_2005_8_76_1_NA_NA,...,0,0,0,0,0,0,0,age_1_m,0,1


In [6]:
[c for c in df.columns]

['ihme_loc_id',
 'geospatial_id',
 'psu',
 'strata',
 'line_id',
 'hh_id',
 'lat',
 'long',
 'lbd_admin2_id',
 'indv_id',
 'year_start',
 'year_end',
 'nid',
 'int_year',
 'int_month',
 'sex_id',
 'age_month',
 'pweight',
 'birth_year',
 'birth_month',
 'int_birth_year_diff_months',
 'age_month_original',
 'int_year_original',
 'int_month_original',
 'any_days_over_30C',
 'child_alive',
 'child_mortality',
 'mean_temperature_cumul',
 'days_over_30C_cumul',
 'total_precipitation_cumul',
 'elevation_cumul',
 'consumption_cumul',
 'consumption_pd_cumul',
 'days_over_24C_monthly_cumul',
 'days_over_25C_monthly_cumul',
 'days_over_26C_monthly_cumul',
 'days_over_27C_monthly_cumul',
 'days_over_28C_monthly_cumul',
 'days_over_29C_monthly_cumul',
 'days_over_30C_monthly_cumul',
 'days_over_31C_monthly_cumul',
 'days_over_32C_monthly_cumul',
 'mean_temperature_monthly_cumul',
 'total_precipitation_monthly_cumul',
 'q75_monthly_cumul',
 'q80_monthly_cumul',
 'q85_monthly_cumul',
 'q90_monthly_c

In [9]:
df["age_group"].unique()

array(['age_1_m', 'age_3_m', 'age_6_m', 'age_12_m', 'age_24_m',
       'age_36_m', 'age_48_m', 'age_60_m'], dtype=object)

In [ ]:
# Previously data contained extra rows following childrens' deaths
df[
    df["indv_id"]
    == df[(df["age_group"] == "age_12_m") & (df["child_mortality"] == 1)][
        "indv_id"
    ].iloc[0]
]

,ihme_loc_id,geospatial_id,psu,strata,line_id,hh_id,lat,long,lbd_admin2_id,indv_id,...,age_3_m,age_6_m,age_12_m,age_24_m,age_36_m,age_48_m,age_60_m,age_group,lower_bound,upper_bound
5598483,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,0,0,0,0,0,0,age_1_m,0,1
10277610,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,1,0,0,0,0,0,0,age_3_m,1,3
20386639,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,1,0,0,0,0,0,age_6_m,3,6
23372903,MWI,100,100,5,3,31,-15.39388,35.34822,92114,21393_100_31_3_2_2003_7_2010_9_86_0_28_6,...,0,0,1,0,0,0,0,age_12_m,6,12


In [14]:
cols_to_check = [
    "child_mortality",
    "consumption_pd_cumul",
    "int_birth_year_diff_months",
    "birth_year",
    "days_over_30C_monthly_cumul",
    "total_precipitation_cumul",
    "lat",
    "long",
]

for c in cols_to_check:
    print(f"{c}: {df[c].isna().sum()} missing values")

child_mortality: 0 missing values
consumption_pd_cumul: 0 missing values
int_birth_year_diff_months: 0 missing values
birth_year: 0 missing values
days_over_30C_monthly_cumul: 0 missing values
total_precipitation_cumul: 0 missing values
lat: 0 missing values
long: 0 missing values


Add median consumption_pd for each country-birth_year

In [ ]:
rt_consumption_df = pd.read_parquet(
    "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/input/ldi/v8/national_estimates.parquet"
)
rt_consumption_df = rt_consumption_df[
    (rt_consumption_df["scenario"] == "reference")
    & (rt_consumption_df["population_percentile"] == 0.5)
]
rt_consumption_df = rt_consumption_df[
    ["ihme_loc_id", "year_id", "location_id", "ldipc"]
]
rt_consumption_df.rename(
    columns={"year_id": "birth_year", "ldipc": "consumption_median_RT"},
    inplace=True,
)

In [17]:
df = df.merge(rt_consumption_df, on=["ihme_loc_id", "birth_year"], how="left")
assert len(df[df["consumption_median_RT"].isna()]) == 0
df["consumption_median_RT"] = df["consumption_median_RT"] / 365

Cut off int_birth_year_diff_months to max 10

In [18]:
df.to_parquet(no_cutoff_path, index=False)

In [19]:
df = df[df["int_birth_year_diff_months"] < 120]
print(f"Number of rows after filtering: {len(df):,}")
df.to_parquet(data_path, index=False)

Number of rows after filtering: 25,972,326
